<a href="https://colab.research.google.com/github/BhavyeNijhawan/swarkavach/blob/main/notebooks/05_voice_cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Voice cloning, your own voice only

> ## STOP. Consent gate.
>
> This notebook clones a voice. It is restricted to **your own voice, with
> your own written consent**, or to a person who has signed the form at
> `docs/CONSENT_FORM.md` in this repository and is present while you run it.
>
> **Do not run it on a recording of anybody else.** Not a friend without the
> signed form, not a public figure, not a YouTube clip, not a sample from a
> dataset. Cloning a voice without consent is impersonation, it is what this
> entire project exists to detect, and doing it here would make the ethics
> section of the report a lie.
>
> The audio produced here stays inside the project. It is not published, not
> uploaded anywhere, and it is deleted at the end of the assessment as the
> consent form promises. The last cell of this notebook deletes the voice
> model and the reference recording.
>
> To proceed, open `docs/CONSENT_FORM.md`, read it, fill it in, and set
> `CONSENT_CONFIRMED = True` in the cell below. Everything after that cell
> refuses to run until you do.

---

## What this produces

The corpus's synthetic cells currently use the repository's own source-filter
synthesiser. That is a deliberate stand-in with the right acoustic properties
(low jitter, over-regular noise floor, soft top band), and it is enough to
train the anti-spoof branch offline. It is not a neural vocoder, and a
detector that only ever sees it has not been tested against the thing it is
actually for.

So: a handful of real cloned-voice calls, from a consenting speaker reading
the project's own generated scam and benign scripts, resampled to 8 kHz and
written back as corpus cells with `audio_source="xtts"`.

**Needs:** a GPU for XTTS-v2. On CPU, XTTS runs at roughly 10x slower than
real time and the notebook falls back to edge-tts, which is a neural voice but
not a clone of you.

**Time:** 10 minutes on a T4, most of it downloading the 1.8 GB XTTS
checkpoint.

**Writes back:** `data/results/cloning_manifest.json`, and the Call JSON files
for the cloned cells. The audio itself is **not** in the download by default;
set `DOWNLOAD_AUDIO = True` only if you understand that it then exists in your
browser's downloads folder as well as in Colab.

In [ ]:
# ---------------------------------------------------------------------------
# Get the repository and install the Colab stack.
# Re-running this cell is safe: if the clone is already on disk it pulls.
# ---------------------------------------------------------------------------

# CHANGE THIS to your own fork before you run anything.
REPO_URL = "https://github.com/BhavyeNijhawan/svarkavach.git"
REPO_DIR = "svk_repo"   # NOT "swarkavach": a directory on sys.path whose name matches the package shadows the real install
BRANCH = "main"

import os
import sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO = BASE / REPO_DIR

if (REPO / "pyproject.toml").exists():
    print(f"{REPO} is already cloned, pulling instead")
    !git -C "{REPO}" pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} "{REPO}"

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("cwd:", os.getcwd())

!pip install -q -r requirements-colab.txt
!pip install -q -e .

import swarkavach
print("swarkavach", swarkavach.__version__, "ready")

## The gate

Read `docs/CONSENT_FORM.md` first. The cell below prints it, then refuses to
continue until `CONSENT_CONFIRMED` is `True` and the identity fields are
filled in.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------------
# Set these three, honestly, or nothing below this cell will run.
# ---------------------------------------------------------------------------
CONSENT_CONFIRMED = False
SPEAKER_NAME = ""          # who is being cloned
CONSENT_DATE = ""          # the date on the signed form, YYYY-MM-DD
DOWNLOAD_AUDIO = False     # keep the cloned audio out of your browser downloads

form = Path("docs/CONSENT_FORM.md")
if form.exists():
    print(form.read_text(encoding="utf-8"))
    print("=" * 72)
else:
    print(f"{form} is missing from this clone. Do not proceed without it.")

if not CONSENT_CONFIRMED:
    raise SystemExit(
        "\nCONSENT_CONFIRMED is False.\n"
        "Read docs/CONSENT_FORM.md, sign it, then set CONSENT_CONFIRMED = True,\n"
        "SPEAKER_NAME and CONSENT_DATE in this cell and run it again.\n"
        "Every cell below this one checks the same flag."
    )

if not (SPEAKER_NAME.strip() and CONSENT_DATE.strip()):
    raise SystemExit(
        "\nSPEAKER_NAME and CONSENT_DATE are empty. The manifest has to record "
        "who consented and when, otherwise the audit trail the consent form "
        "promises does not exist."
    )


def require_consent():
    """Called at the top of every cell that touches voice data."""
    if not CONSENT_CONFIRMED:
        raise RuntimeError("consent gate not passed, see the first code cell")
    if not (SPEAKER_NAME.strip() and CONSENT_DATE.strip()):
        raise RuntimeError("speaker identity or consent date missing")


require_consent()
print(f"Consent recorded for {SPEAKER_NAME}, form dated {CONSENT_DATE}.")
print(f"Cloned audio download is {'ON' if DOWNLOAD_AUDIO else 'OFF'}.")

## The reference clip

XTTS-v2 wants 6 to 15 seconds of clean speech. Ten seconds of normal talking,
in a quiet room, no music, no other voices. Read anything; the content does
not matter, only the voice.

Two ways to provide it. The recorder below uses the browser microphone and is
the quicker path. If it fails (some browsers block microphone access in an
iframe), upload a wav or mp3 instead.

In [ ]:
require_consent()

import base64
import subprocess
from pathlib import Path

REF_DIR = Path("data/voice_ref")
REF_DIR.mkdir(parents=True, exist_ok=True)
REF_WAV = REF_DIR / "reference.wav"
RECORD_SECONDS = 10

RECORDER_JS = """
async function recordBlob(sec) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  const div = document.createElement('div');
  div.textContent = 'recording ' + sec + ' seconds...';
  document.body.appendChild(div);
  await new Promise(r => setTimeout(r, sec * 1000));
  rec.stop();
  await new Promise(r => rec.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  div.textContent = 'done';
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  const bytes = new Uint8Array(buf);
  let bin = '';
  for (let i = 0; i < bytes.length; i++) bin += String.fromCharCode(bytes[i]);
  return btoa(bin);
}
"""

recorded = False
try:
    from google.colab import output
    from IPython.display import Javascript, display

    display(Javascript(RECORDER_JS))
    print(f"Speak for {RECORD_SECONDS} seconds when the message appears.")
    b64 = output.eval_js(f"recordBlob({RECORD_SECONDS})", timeout_sec=RECORD_SECONDS + 30)
    raw = REF_DIR / "reference_raw.webm"
    raw.write_bytes(base64.b64decode(b64))
    rc = subprocess.run(
        f'ffmpeg -y -loglevel error -i "{raw}" -ac 1 -ar 22050 "{REF_WAV}"',
        shell=True).returncode
    recorded = rc == 0 and REF_WAV.exists()
    raw.unlink(missing_ok=True)
    print(f"recorded {REF_WAV} ({REF_WAV.stat().st_size / 1024:.0f} kB)" if recorded
          else "ffmpeg conversion failed")
except Exception as exc:
    print(f"browser recording unavailable ({type(exc).__name__}: {exc})")

if not recorded:
    print("\nUpload a wav or mp3 of the consenting speaker instead.")
    try:
        from google.colab import files

        up = files.upload()
        name = next(iter(up))
        subprocess.run(
            f'ffmpeg -y -loglevel error -i "{name}" -ac 1 -ar 22050 "{REF_WAV}"',
            shell=True)
        recorded = REF_WAV.exists()
        print(f"converted {name} to {REF_WAV}")
    except Exception as exc:
        print(f"upload failed: {type(exc).__name__}: {exc}")

if recorded:
    from swarkavach.audioio import read_audio

    x, sr = read_audio(str(REF_WAV), sr=22050)
    print(f"reference: {len(x) / sr:.1f} s at {sr} Hz")
    if len(x) / sr < 5:
        print("Under 5 seconds. XTTS will clone it badly. Record again.")
    try:
        from IPython.display import Audio, display

        display(Audio(str(REF_WAV)))
    except Exception:
        pass
else:
    print("\nNo reference audio. The XTTS path below will be skipped and the "
          "edge-tts fallback used instead, which is a neural voice but not "
          "a clone of anybody.")

## The scripts to read

Straight from the generated corpus, so the cloned cells line up with the rest
of the evaluation grid: the same scenarios, the same gold BIO, the same
dialogue acts. A balanced handful of scam and benign calls, since the point of
the cloned cells is to fill both `clonedxscam` and `clonedxbenign`.

Nothing in these scripts is real. No real bank, no real UPI handle, no real
phone number, no real case number. That is a property of the generator, and it
is why these are safe to read aloud into a microphone.

In [ ]:
require_consent()

import subprocess
import sys
from pathlib import Path

if not any(Path("data/corpus/calls").glob("*.json")):
    print("no corpus, generating one")
    subprocess.run([sys.executable, "-m", "swarkavach.cli", "gen-corpus",
                    "--n", "480", "--audio"], check=False)

from swarkavach.corpus.generator import load_corpus

CALLS = load_corpus()
N_CLONE = 6                       # calls to re-render with the cloned voice

scam = [c for c in CALLS if c.label_scam][:N_CLONE // 2]
benign = [c for c in CALLS if not c.label_scam][:N_CLONE - len(scam)]
TO_CLONE = scam + benign

print(f"{len(TO_CLONE)} calls selected:")
for c in TO_CLONE:
    print(f"  {c.call_id}  {'scam  ' if c.label_scam else 'benign'}  "
          f"{c.scenario:<24} {len(c.turns)} turns, {c.duration:.1f}s")

print("\nfirst call, caller turns:")
for t in TO_CLONE[0].caller_turns()[:4]:
    print(f"  [{t.act}] {t.text}")

## XTTS-v2

Coqui's XTTS-v2 does zero-shot cloning: it conditions on a reference clip
rather than being fine-tuned on it, which is why ten seconds is enough and why
there is no trained speaker model left behind. It supports Hindi, which
matters for the code-mixed scripts, though it will read romanised Hinglish as
English-ish and the pronunciation will be approximate. That is acceptable
here, because the anti-spoof branch is listening to how the waveform was made,
not to whether the accent is right.

The original `TTS` package is archived; `coqui-tts` is the maintained fork and
imports under the same name. Both are tried.

`COQUI_TOS_AGREED=1` accepts the Coqui Public Model Licence non-interactively.
Read it before setting it: it restricts the model to non-commercial use, which
this project is.

In [ ]:
require_consent()

import os

os.environ["COQUI_TOS_AGREED"] = "1"

XTTS_OK = False
tts = None

if not recorded:
    print("no reference clip, skipping XTTS")
else:
    for pkg in ("coqui-tts", "TTS"):
        print(f"trying pip install {pkg}")
        rc = os.system(f"pip install -q {pkg} 2>/dev/null")
        try:
            from TTS.api import TTS as CoquiTTS

            print(f"  imported after installing {pkg}")
            break
        except Exception as exc:
            print(f"  import failed after {pkg}: {type(exc).__name__}")
            CoquiTTS = None

    if CoquiTTS is not None:
        try:
            import torch

            dev = "cuda" if torch.cuda.is_available() else "cpu"
            if dev == "cpu":
                print("No GPU. XTTS on CPU runs at roughly 10x slower than real time; "
                      "this will take a while, and the edge-tts fallback below is "
                      "the sensible choice if you are short on time.")
            tts = CoquiTTS("tts_models/multilingual/multi-dataset/xtts_v2").to(dev)
            XTTS_OK = True
            print(f"XTTS-v2 loaded on {dev}")
        except Exception as exc:
            print(f"XTTS failed to load: {type(exc).__name__}: {exc}")
            print("Plan B section 4: fall through to edge-tts or Piper below.")

## Render, resample to 8 kHz, write the corpus cells

XTTS synthesises at 24 kHz. The corpus is 8 kHz, because the whole system is
built for telephone bandwidth, so each turn is resampled through
`swarkavach.audioio.resample` (the same windowed-sinc resampler the rest of
the project uses) and the turns are joined with realistic gaps by
`concat_with_gaps`, which also returns the per-turn time spans that the Call
JSON needs.

Downsampling to 8 kHz removes a lot of what makes the clone detectable, which
is exactly the point: this is the hard case for the anti-spoof branch and it
is the case that actually reaches a phone.

In [ ]:
require_consent()

import time
from dataclasses import replace
from pathlib import Path

import numpy as np

from swarkavach.audioio import concat_with_gaps, read_audio, resample, write_wav
from swarkavach.config import TARGET_SR
from swarkavach.schema import Call

OUT_AUDIO = Path("data/corpus/audio")
OUT_CALLS = Path("data/corpus/calls")
OUT_AUDIO.mkdir(parents=True, exist_ok=True)
OUT_CALLS.mkdir(parents=True, exist_ok=True)

CLONED = []
BACKEND = "xtts" if XTTS_OK else None


def synth_xtts(text: str, out_path: Path, language: str = "hi") -> bool:
    try:
        tts.tts_to_file(text=text, speaker_wav=str(REF_WAV),
                        language=language, file_path=str(out_path))
        return out_path.exists()
    except Exception as exc:
        print(f"    xtts failed on a turn: {type(exc).__name__}: {exc}")
        return False


if XTTS_OK:
    tmp = Path("/tmp/swk_tts")
    tmp.mkdir(parents=True, exist_ok=True)
    for call in TO_CLONE:
        t0 = time.time()
        segments = []
        ok = True
        for turn in call.turns:
            piece = tmp / f"{call.call_id}_{turn.index}.wav"
            if not synth_xtts(turn.text, piece):
                ok = False
                break
            x, sr = read_audio(str(piece), sr=None)
            segments.append(resample(np.asarray(x, dtype=np.float32), int(sr), TARGET_SR)
                            if int(sr) != TARGET_SR else np.asarray(x, dtype=np.float32))
            piece.unlink(missing_ok=True)
        if not ok or not segments:
            print(f"  {call.call_id}: skipped")
            continue

        signal, spans = concat_with_gaps(segments, TARGET_SR, gap_s=0.35)
        new_id = f"{call.call_id}_xtts"
        wav_path = OUT_AUDIO / f"{new_id}.wav"
        write_wav(str(wav_path), signal, TARGET_SR)

        turns = []
        for turn, (a, b) in zip(call.turns, spans):
            turns.append(replace(turn, t_start=round(float(a), 3), t_end=round(float(b), 3)))

        cloned = Call(
            call_id=new_id, turns=turns,
            label_scam=call.label_scam, label_voice="synthetic",
            scenario=call.scenario, speaker_id=f"clone_{SPEAKER_NAME[:8] or 'anon'}",
            split=call.split, audio_path=str(wav_path), sample_rate=TARGET_SR,
            channel="clean", audio_source="xtts",
            meta={**dict(call.meta), "cloned_from": call.call_id,
                  "tts_backend": "xtts_v2", "consent_date": CONSENT_DATE,
                  "consent_speaker": SPEAKER_NAME,
                  "delete_by": "end of assessment, see docs/CONSENT_FORM.md"},
        )
        cloned.save(str(OUT_CALLS / f"{new_id}.json"))
        CLONED.append({"call_id": new_id, "source": call.call_id,
                       "scam": int(call.label_scam), "scenario": call.scenario,
                       "duration": round(float(len(signal)) / TARGET_SR, 2),
                       "backend": "xtts_v2", "wav": str(wav_path)})
        print(f"  {new_id}  {len(signal) / TARGET_SR:.1f}s  ({time.time() - t0:.0f}s)")

    print(f"\n{len(CLONED)} cloned calls written")
else:
    print("XTTS not available, nothing rendered here. See the fallback cell.")

## Fallbacks, exactly as Plan B section 4 specifies

If XTTS will not install or will not run, the anti-spoof branch does not
actually need a clone of a specific person. It needs synthetic against human,
and any neural text-to-speech provides that. Two lighter options, in order of
how little they ask of the machine:

**edge-tts.** Microsoft's neural voices over a network call. No model
download, no GPU, and it has Hindi and Indian English voices, which suits the
code-mixed scripts. The repository already implements this path in
`corpus.tts.render_call_audio(call, backend="edge")`, with a different voice
per speaker, turn timings written back onto the Call, and an automatic drop to
the source-filter synthesiser if the network is unavailable. Use it rather
than writing a second copy here. It is not a clone of you, so these cells are
labelled `audio_source="edge"` and the manifest records the substitution
rather than quietly pretending they are clones.

**Piper.** A small ONNX model that runs on a CPU in real time. Useful if the
runtime has no network access to Microsoft's endpoint. English voices only in
the default set, so the Hinglish comes out anglicised.

Both fill the same evaluation cells. Neither is a voice clone, and the report
must not describe them as one.

In [ ]:
require_consent()

import copy
import os
from pathlib import Path

from swarkavach.config import TARGET_SR
from swarkavach.corpus.tts import AVAILABLE_BACKENDS, backend_notes, render_call_audio

if CLONED:
    print(f"{len(CLONED)} XTTS calls already rendered, fallback not needed.")
    edge_ok = True
else:
    print("no XTTS output, trying edge-tts through the repository's own renderer")
    os.system("pip install -q edge-tts 2>/dev/null")
    print("back ends this runtime can use:", AVAILABLE_BACKENDS())
    for name, note in backend_notes().items():
        print(f"  {name:<6} {note}")

    edge_ok = False
    for call in TO_CLONE:
        try:
            # A copy, because render_call_audio rewrites turn timings in place
            # and the original corpus call must keep its own.
            cloned = copy.deepcopy(call)
            cloned.call_id = f"{call.call_id}_edge"
            cloned.label_voice = "synthetic"
            cloned.speaker_id = "edge_neural"
            cloned.meta = {**dict(call.meta), "cloned_from": call.call_id,
                           "not_a_clone": True,
                           "note": "neural TTS, not a clone of any person"}
            out = Path("data/corpus/audio") / f"{cloned.call_id}.wav"
            signal, info = render_call_audio(cloned, backend="edge",
                                             out_path=out, sr=TARGET_SR)
            if info["backend"] != "edge":
                print(f"  fell back to {info['backend']}: {info['fallback_reason']}")
                break
            cloned.meta["tts_backend"] = f"edge-tts {info['detail'].get('caller_voice')}"
            cloned.save(str(Path("data/corpus/calls") / f"{cloned.call_id}.json"))
            CLONED.append({"call_id": cloned.call_id, "source": call.call_id,
                           "scam": int(call.label_scam), "scenario": call.scenario,
                           "duration": info["duration_s"],
                           "backend": f"edge-tts {info['detail'].get('caller_voice')}",
                           "wav": str(out)})
            edge_ok = True
            print(f"  {cloned.call_id}  {info['duration_s']:.1f}s")
        except Exception as exc:
            print(f"  edge-tts failed on {call.call_id}: {type(exc).__name__}: {exc}")
            break

    if not edge_ok:
        print("\nedge-tts did not work either. Piper is the last option:")
        print("  !pip install -q piper-tts")
        print("  !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/"
              "en/en_US/lessac/medium/en_US-lessac-medium.onnx")
        print("  !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/"
              "en/en_US/lessac/medium/en_US-lessac-medium.onnx.json")
        print("  then: echo 'text' | piper -m en_US-lessac-medium.onnx -f out.wav")
        print("\nIf none of the three work, the corpus keeps its source-filter")
        print("synthetic cells. Notebooks 02 and 04 run unchanged; the report says")
        print("the synthetic cells are simulated rather than neural, which is true")
        print("and is already the offline default.")

## Does the anti-spoof branch notice

A quick sanity check rather than an evaluation. If the RawNet checkpoint from
notebook 02 is on disk, score the new cells with it. A cloned or neural-TTS
cell should score higher on P(synthetic) than the human cells it was rendered
from.

If it does not, that is a finding worth writing down: a model trained on
ASVspoof 2019 attacks (2019 vintage vocoders) is being asked about XTTS-v2
(2023), and the generalisation gap in notebook 02 predicts exactly this kind
of miss.

In [ ]:
require_consent()

from pathlib import Path

import numpy as np

CHECK = {}
ckpt = Path("data/models/antispoof_rawnet.pt")
if CLONED and ckpt.exists():
    from swarkavach.antispoof.rawnet import load_rawnet
    from swarkavach.audioio import read_audio

    net = load_rawnet(str(ckpt))
    if net is not None:
        by_id = {c.call_id: c for c in CALLS}
        for row in CLONED:
            try:
                x, sr = read_audio(row["wav"], sr=TARGET_SR)
                p_new = float(net.score(x, sr))
                src = by_id.get(row["source"])
                p_src = None
                if src and src.audio_path and Path(src.audio_path).exists():
                    xs, ss = read_audio(src.audio_path, sr=TARGET_SR)
                    p_src = float(net.score(xs, ss))
                CHECK[row["call_id"]] = {"p_synthetic": round(p_new, 4),
                                         "p_source": round(p_src, 4) if p_src else None,
                                         "backend": row["backend"]}
                print(f"  {row['call_id']:<28} P(synthetic) {p_new:.3f}"
                      + (f"   source call {p_src:.3f}" if p_src is not None else ""))
            except Exception as exc:
                print(f"  {row['call_id']}: {type(exc).__name__}: {exc}")
    else:
        print("checkpoint did not load")
elif not ckpt.exists():
    print("no RawNet checkpoint on disk. Run notebook 02 or unzip its download here.")
else:
    print("nothing was cloned, nothing to check")

## Export

The manifest goes back to the laptop. The audio does not, unless you turned
`DOWNLOAD_AUDIO` on, because a cloned voice on a laptop's Downloads folder is
one accidental drag away from being somewhere it should not be. The Call JSON
files travel, so the local corpus knows the cells exist; if you want the audio
locally, re-render it there or set the flag deliberately.

In [ ]:
# ---------------------------------------------------------------------------
# Export helpers. Each notebook carries its own copy so it can run on its own.
#
# Everything written here lands in data/results/ with "source": "colab", a UTC
# timestamp and the GPU name, which is what the local console's provenance
# table reads to tell your laptop numbers apart from your Colab numbers.
# ---------------------------------------------------------------------------
import json
import platform
import zipfile
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK = "05_voice_cloning"
RESULTS = Path("data/results")
RESULTS.mkdir(parents=True, exist_ok=True)


def gpu_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return "none (CPU runtime)"


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


def colab_stamp(notes: str = "") -> dict:
    try:
        import torch

        tv = str(torch.__version__)
    except Exception:
        tv = "not installed"
    return {
        "source": "colab",
        "generated": utc_now(),
        "gpu": gpu_name(),
        "torch": tv,
        "python": platform.python_version(),
        "notebook": NOTEBOOK,
        "notes": notes,
    }


WRITTEN = []


def write_result(name: str, obj: dict, notes: str = "") -> Path:
    """Write one result file, stamped, and remember it for the zip."""
    doc = dict(obj)
    doc.update(colab_stamp(notes))
    path = RESULTS / name
    path.write_text(json.dumps(doc, indent=2, default=float), encoding="utf-8")
    if name not in WRITTEN:
        WRITTEN.append(name)
    print(f"wrote {path}  ({path.stat().st_size / 1024:.1f} kB)")
    return path


def update_provenance(entries) -> Path:
    """Merge these Colab entries into data/results/provenance.json.

    Entries for other artifacts are kept, so a local `swarkavach evaluate` run
    and a Colab run can sit in the same table. Unzip the Colab results AFTER
    the local evaluate, otherwise the local file overwrites this one.
    """
    path = RESULTS / "provenance.json"
    doc = {"entries": []}
    if path.exists():
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    new_names = {e["artifact"] for e in entries}
    kept = [e for e in doc.get("entries", []) if e.get("artifact") not in new_names]
    doc["entries"] = kept + list(entries)
    doc["generated"] = utc_now()
    doc["host"] = platform.platform()
    doc["gpu"] = gpu_name()
    path.write_text(json.dumps(doc, indent=2), encoding="utf-8")
    if "provenance.json" not in WRITTEN:
        WRITTEN.append("provenance.json")
    print(f"provenance now lists {len(doc['entries'])} artifacts")
    return path


def export_zip(extra_paths=(), zip_name: str = None) -> Path:
    """Zip the result files plus anything extra, then hand it to the browser."""
    out = Path(zip_name or f"swarkavach_{NOTEBOOK}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in WRITTEN:
            f = RESULTS / name
            if f.exists():
                z.write(f, arcname=f"data/results/{f.name}")
        for extra in extra_paths:
            p = Path(extra)
            if p.exists():
                z.write(p, arcname=p.as_posix())
            else:
                print(f"  skipped missing {p}")

    names = zipfile.ZipFile(out).namelist()
    print(f"\n{out.name}  {out.stat().st_size / 1024:.1f} kB")
    for n in names:
        print("   ", n)

    try:
        from google.colab import files

        files.download(str(out))
        print("\nDownload started. Unzip it over your local project root.")
    except Exception as exc:
        print(f"\nnot running on Colab ({type(exc).__name__}), zip is at {out.resolve()}")
    return out

In [ ]:
require_consent()

from pathlib import Path

manifest = {
    "consent": {
        "confirmed": bool(CONSENT_CONFIRMED),
        "speaker": SPEAKER_NAME,
        "form_date": CONSENT_DATE,
        "form": "docs/CONSENT_FORM.md",
        "delete_by": "end of assessment, no later than 31 December 2026",
    },
    "backend": CLONED[0]["backend"] if CLONED else "none",
    "n_cloned": len(CLONED),
    "calls": CLONED,
    "antispoof_check": CHECK,
    "sample_rate": TARGET_SR,
    "reference_deleted": False,
    "note": ("cells rendered by edge-tts or Piper are neural TTS, not a clone of "
             "any particular person; the per-call meta records which."),
}
write_result("cloning_manifest.json", manifest,
             notes=f"{len(CLONED)} cells rendered with consent from {SPEAKER_NAME}")

update_provenance([{
    "artifact": "cloning_manifest.json", "source": "colab", "generated": utc_now(),
    "notes": (f"{len(CLONED)} consented cloned-voice cells, "
              f"{manifest['backend']}, consent dated {CONSENT_DATE}"),
}])

extras = [Path(f"data/corpus/calls/{row['call_id']}.json") for row in CLONED]
if DOWNLOAD_AUDIO:
    extras += [Path(row["wav"]) for row in CLONED]
    print("DOWNLOAD_AUDIO is on: the cloned wav files are in the zip.")
else:
    print("DOWNLOAD_AUDIO is off: only the Call JSON travels, not the audio.")

export_zip(extra_paths=extras)

## Delete the voice model and the reference recording

The consent form says the recording and anything derived from it are deleted.
XTTS-v2 is zero-shot, so there is no fine-tuned speaker model to delete, but
there is a 1.8 GB checkpoint in the cache and there is the reference clip, and
both come out now.

Run this cell before you close the session. If the session dies first, Colab
deletes the disk anyway, which is not a plan but is a backstop. If you mounted
Drive, check Drive by hand: nothing here writes voice data there, but check
anyway.

In [ ]:
import shutil
from pathlib import Path

removed = []

# 1. the reference recording and anything beside it
ref_dir = Path("data/voice_ref")
if ref_dir.exists():
    for f in ref_dir.rglob("*"):
        if f.is_file():
            removed.append(str(f))
    shutil.rmtree(ref_dir, ignore_errors=True)

# 2. the XTTS model cache
for cache in (Path.home() / ".local/share/tts", Path("/root/.local/share/tts"),
              Path.home() / ".cache/tts"):
    if cache.exists():
        size = sum(f.stat().st_size for f in cache.rglob("*") if f.is_file()) / 1e9
        shutil.rmtree(cache, ignore_errors=True)
        removed.append(f"{cache} ({size:.2f} GB)")

# 3. any temporary synthesis pieces
for tmpdir in (Path("/tmp/swk_tts"), Path("/tmp/swk_edge")):
    if tmpdir.exists():
        shutil.rmtree(tmpdir, ignore_errors=True)
        removed.append(str(tmpdir))

# 4. drop the in-memory model
try:
    del tts
except Exception:
    pass
try:
    import torch

    torch.cuda.empty_cache()
except Exception:
    pass

print("deleted:")
for r in removed:
    print("  -", r)
if not removed:
    print("  nothing left to delete")

print(f"\nreference clip present: {Path('data/voice_ref/reference.wav').exists()}")
print(f"cloned audio still in data/corpus/audio: "
      f"{len(list(Path('data/corpus/audio').glob('*_xtts.wav')) + list(Path('data/corpus/audio').glob('*_edge.wav')))} files")

## Before you close this notebook

The cloned audio stays inside the project. Concretely:

- Do not commit the wav files. `.gitignore` already excludes `*.wav` and
  `data/corpus/audio/`; do not use `git add -f` on them.
- Do not attach them to a report, a slide deck, an email or a message.
  Spectrograms and waveform plots are fine. The audio is not.
- Do not upload them to any dataset, model hub, or file-sharing service.
- Delete them at the end of the assessment, along with everything derived from
  them, as the consent form promises. That includes the local copy if you set
  `DOWNLOAD_AUDIO = True`.
- If the speaker withdraws, delete everything within seven days and say so.
  They do not have to give a reason and they do not have to be asked twice.

The reason this project can defend having a cloning notebook at all is that it
is a detector, the cloning is consented, the scope is one person's own voice,
and the audio is destroyed. Break any of those four and the ethics section
stops being true.